# 🧠 04. Explainability & Counterfactuals

This notebook demonstrates how to generate **Counterfactual Explanations** using the DiCE integration.

**The Question:** "What minimal change to the microbiome composition would flip this 'CRC' prediction to 'Healthy'?"

In [1]:
import microfactual as mf
from microfactual import MicrobiomeClassifier, MicrobiomeDataset

## 1. Setup Model

We train a model first. For DiCE, we'll keep the model simple to ensure faster explanation generation.

In [2]:
dataset = MicrobiomeDataset.from_files(
    abundance_file="../datasets/abundance_crc.txt",
    metadata_file="../datasets/metadata_crc.txt",
    target_column="Group",
)

# using Logistic Regression for faster counterfactual generation
clf = MicrobiomeClassifier(algorithm="logistic", max_iter=1000)
clf.fit(dataset.X, dataset.y)

,algorithm,'logistic'
,preprocessing,'auto'


## 2. Initialize Explainer

We use `DiCEExplainer`.

In [3]:
explainer = mf.DiCEExplainer(
    model=clf, background_data=dataset.X, target_column="Group", target_data=dataset.y
)

## 3. Generate Explanation

Let's take a sample that was predicted as **CRC (Class 1)** and ask how to make it **Healthy (Class 0)**.

In [4]:
# Find a CRC sample (where y=1)
query_idx = dataset.y[dataset.y == 1].index[0]
query_instance = dataset.X.loc[[query_idx]]

print(f"Explaining Sample: {query_idx}")
print(f"Current Prediction: {clf.predict(query_instance)[0]}")

Explaining Sample: CCIS27304052ST-3-0
Current Prediction: 1


In [5]:
# Generate 3 Counterfactuals
explanation = explainer.explain(
    query_instance,
    total_CFs=3,
    desired_class=0,  # Target class 0 (Healthy)
)

100%|██████████| 1/1 [00:02<00:00,  2.82s/it]


## 4. Visualize Results

Show only the changes required.

In [6]:
explanation.visualize_as_dataframe(show_only_changes=True)

Query instance (original outcome : 1)


,Methanoculleus marisnigri [1],Methanococcoides burtonii [10],Thermococcus kodakarensis [100],Victivallis vadensis [1000],Gemmatimonas aurantiaca [1001],Fibrobacter succinogenes [1002],unnamed Verrucomicrobiae bacterium DG1235 [1003],Opitutus terrae [1004],Coraliomargarita akajimensis [1005],Verrucomicrobium spinosum [1006],...,Marinithermus hydrothermalis [991],Deinococcus radiodurans [992],Deinococcus deserti [993],Deinococcus geothermalis [994],Deinococcus proteolyticus [995],Deinococcus maricopensis [996],Truepera radiovictrix [997],Rhodothermus marinus [998],Salinibacter ruber [999],Group
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1



Diverse Counterfactual set (new outcome: 0)


,Methanoculleus marisnigri [1],Methanococcoides burtonii [10],Thermococcus kodakarensis [100],Victivallis vadensis [1000],Gemmatimonas aurantiaca [1001],Fibrobacter succinogenes [1002],unnamed Verrucomicrobiae bacterium DG1235 [1003],Opitutus terrae [1004],Coraliomargarita akajimensis [1005],Verrucomicrobium spinosum [1006],...,Marinithermus hydrothermalis [991],Deinococcus radiodurans [992],Deinococcus deserti [993],Deinococcus geothermalis [994],Deinococcus proteolyticus [995],Deinococcus maricopensis [996],Truepera radiovictrix [997],Rhodothermus marinus [998],Salinibacter ruber [999],Group
0,-,-,-,-,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,0.0
0,-,-,-,-,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,0.0
0,-,-,-,-,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,0.0
